In [1]:
from numpy import array
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense
import numpy as np
import pandas as pd
import tensorflow as tf

In [2]:
# split a univariate sequence into samples
def split_sequence(sequence, n_steps):
    X, y = list(), list()
    for i in range(len(sequence)):
        # find the end of this pattern
        end_ix = i + n_steps
        # check if we are beyond the sequence
        if end_ix > len(sequence)-1:
            break
        # gather input and output parts of the pattern
        seq_x, seq_y = sequence[i:end_ix], sequence[end_ix]
        X.append(seq_x)
        y.append(seq_y)
    return array(X), array(y)

In [3]:
# define input sequence
parser = (lambda x:datetime.datetime.strptime(x, '%Y.%m.%d')) 
df = pd.read_csv('sp_beaches_update.csv', parse_dates=['Date'])
df = df.sort_values(by=['Date'])
df=df.loc[~df['Enterococcus'].isnull()]
#remover a praia do Leste, da cidade de iguape, pois esta praia sumiu por erosão em 2012
#remover a Lagoa Prumirim, da cidade de Ubatuba, pois esta praia possui somente 3 medições
df = df.loc[df['Beach']!='DO LESTE'].loc[df['Beach']!='LAGOA PRUMIRIM']
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 69325 entries, 0 to 66738
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   City          69325 non-null  object        
 1   Beach         69325 non-null  object        
 2   Date          69325 non-null  datetime64[ns]
 3   Enterococcus  69325 non-null  int64         
dtypes: datetime64[ns](1), int64(1), object(2)
memory usage: 2.6+ MB


In [4]:
df.loc[df['City']=='UBATUBA'].loc[df['Beach']=='GRANDE']

,City,Beach,Date,Enterococcus
7614,UBATUBA,GRANDE,2012-01-03,5
7615,UBATUBA,GRANDE,2012-01-08,42
7616,UBATUBA,GRANDE,2012-01-15,92
7617,UBATUBA,GRANDE,2012-01-22,16
7618,UBATUBA,GRANDE,2012-01-29,9
...,...,...,...,...
69094,UBATUBA,GRANDE,2020-10-05,31
69095,UBATUBA,GRANDE,2020-10-12,15
69096,UBATUBA,GRANDE,2020-10-19,3
69097,UBATUBA,GRANDE,2020-10-26,12


In [7]:
# selecionar praias com medições semanal
df_freq = pd.read_csv('frequencia_praias.csv')
df_freq = df_freq.loc[df_freq['Beach']!='DO LESTE'].loc[df['Beach']!='LAGOA PRUMIRIM']
df_freq = df_freq.loc[df_freq['Frequency']=='SEMANAL']
df_freq

,City,Beach,Frequency
9,BERTIOGA,BORACÉIA - COL. MARISTA,SEMANAL
10,BERTIOGA,BORACÉIA - SUL,SEMANAL
11,BERTIOGA,GUARATUBA,SEMANAL
12,BERTIOGA,SÃO LOURENÇO (JUNTO AO MORRO),SEMANAL
13,BERTIOGA,SÃO LOURENÇO (RUA 2),SEMANAL
...,...,...,...
169,UBATUBA,DURA,SEMANAL
170,UBATUBA,LAGOINHA (R. ENGENHO VELHO),SEMANAL
171,UBATUBA,LAGOINHA (CAMPING),SEMANAL
172,UBATUBA,SAPÉ,SEMANAL


In [9]:
n_steps = 16
for cidade in df_freq.City.unique():
    #print(df2.loc[df2['City']==cidade].Beach.unique())
    for praia in df_freq.loc[df_freq['City']==cidade].Beach.unique():
        print(f'iniciando: {cidade} - {praia}')
        df_beach = df.loc[df['City']==cidade].loc[df['Beach']==praia][['Date','Enterococcus']]
        df_beach.columns = ['ds', 'y']
        df_beach.set_index('ds', inplace=True)
        X, y = split_sequence(df_beach.values, n_steps)
        n_features = 1
        X = X.reshape((X.shape[0], X.shape[1], n_features))
        model = Sequential()
        model.add(LSTM(104, activation='relu', return_sequences=True, input_shape=(n_steps, n_features)))
        model.add(LSTM(104, activation='relu'))
        model.add(Dense(1))
        model.compile(optimizer='adam', loss='mse')
        print('Treinando modelo: '+cidade+'-'+praia)
        
        with tf.device('/GPU:0'): 
            R = model.fit(X, y, epochs=200, verbose=0)
        loss=round(R.history['loss'][-1])
        model.save('model'+cidade+'-'+praia+'.h5')        
        print('Modelo Treinado: '+cidade+'-'+praia+'- MSE:'+str(loss))
        

iniciando: BERTIOGA - BORACÉIA - COL. MARISTA
Treinando modelo: BERTIOGA-BORACÉIA - COL. MARISTA
Modelo Treinado: BERTIOGA-BORACÉIA - COL. MARISTA- MSE:646
iniciando: BERTIOGA - BORACÉIA - SUL
Treinando modelo: BERTIOGA-BORACÉIA - SUL
Modelo Treinado: BERTIOGA-BORACÉIA - SUL- MSE:1666
iniciando: BERTIOGA - GUARATUBA
Treinando modelo: BERTIOGA-GUARATUBA
Modelo Treinado: BERTIOGA-GUARATUBA- MSE:791
iniciando: BERTIOGA - SÃO LOURENÇO (JUNTO AO MORRO)
Treinando modelo: BERTIOGA-SÃO LOURENÇO (JUNTO AO MORRO)
Modelo Treinado: BERTIOGA-SÃO LOURENÇO (JUNTO AO MORRO)- MSE:964
iniciando: BERTIOGA - SÃO LOURENÇO (RUA 2)
Treinando modelo: BERTIOGA-SÃO LOURENÇO (RUA 2)
Modelo Treinado: BERTIOGA-SÃO LOURENÇO (RUA 2)- MSE:1399
iniciando: BERTIOGA - ENSEADA - INDAIÁ
Treinando modelo: BERTIOGA-ENSEADA - INDAIÁ
Modelo Treinado: BERTIOGA-ENSEADA - INDAIÁ- MSE:2416
iniciando: BERTIOGA - ENSEADA - VISTA LINDA
Treinando modelo: BERTIOGA-ENSEADA - VISTA LINDA
Modelo Treinado: BERTIOGA-ENSEADA - VISTA LINDA- 

Modelo Treinado: ILHABELA-CURRAL- MSE:3207
iniciando: ITANHAÉM - CAMPOS ELÍSEOS
Treinando modelo: ITANHAÉM-CAMPOS ELÍSEOS
Modelo Treinado: ITANHAÉM-CAMPOS ELÍSEOS- MSE:11727
iniciando: ITANHAÉM - SUARÃO
Treinando modelo: ITANHAÉM-SUARÃO
Modelo Treinado: ITANHAÉM-SUARÃO- MSE:9508
iniciando: ITANHAÉM - SUARÃO - AFPESP
Treinando modelo: ITANHAÉM-SUARÃO - AFPESP
Modelo Treinado: ITANHAÉM-SUARÃO - AFPESP- MSE:4578
iniciando: ITANHAÉM - PARQUE BALNEÁRIO
Treinando modelo: ITANHAÉM-PARQUE BALNEÁRIO
Modelo Treinado: ITANHAÉM-PARQUE BALNEÁRIO- MSE:10171
iniciando: ITANHAÉM - CENTRO
Treinando modelo: ITANHAÉM-CENTRO
Modelo Treinado: ITANHAÉM-CENTRO- MSE:11902
iniciando: ITANHAÉM - PRAIA DOS PESCADORES
Treinando modelo: ITANHAÉM-PRAIA DOS PESCADORES
Modelo Treinado: ITANHAÉM-PRAIA DOS PESCADORES- MSE:6872
iniciando: ITANHAÉM - SONHO
Treinando modelo: ITANHAÉM-SONHO
Modelo Treinado: ITANHAÉM-SONHO- MSE:6049
iniciando: ITANHAÉM - JARDIM CIBRATEL
Treinando modelo: ITANHAÉM-JARDIM CIBRATEL
Modelo Trei

Modelo Treinado: SÃO SEBASTIÃO-CAMBURI- MSE:9413
iniciando: SÃO SEBASTIÃO - BALEIA
Treinando modelo: SÃO SEBASTIÃO-BALEIA
Modelo Treinado: SÃO SEBASTIÃO-BALEIA- MSE:2904
iniciando: SÃO SEBASTIÃO - SAÍ
Treinando modelo: SÃO SEBASTIÃO-SAÍ
Modelo Treinado: SÃO SEBASTIÃO-SAÍ- MSE:13640
iniciando: SÃO SEBASTIÃO - PRETA
Treinando modelo: SÃO SEBASTIÃO-PRETA
Modelo Treinado: SÃO SEBASTIÃO-PRETA- MSE:8482
iniciando: SÃO SEBASTIÃO - JUQUEÍ (TRAV. SIMÃO FAUSTINO)
Treinando modelo: SÃO SEBASTIÃO-JUQUEÍ (TRAV. SIMÃO FAUSTINO)
Modelo Treinado: SÃO SEBASTIÃO-JUQUEÍ (TRAV. SIMÃO FAUSTINO)- MSE:5218
iniciando: SÃO SEBASTIÃO - JUQUEÍ (R. CRISTIANA)
Treinando modelo: SÃO SEBASTIÃO-JUQUEÍ (R. CRISTIANA)
Modelo Treinado: SÃO SEBASTIÃO-JUQUEÍ (R. CRISTIANA)- MSE:10315
iniciando: SÃO SEBASTIÃO - UNA
Treinando modelo: SÃO SEBASTIÃO-UNA
Modelo Treinado: SÃO SEBASTIÃO-UNA- MSE:5239
iniciando: SÃO SEBASTIÃO - ENGENHO
Treinando modelo: SÃO SEBASTIÃO-ENGENHO
Modelo Treinado: SÃO SEBASTIÃO-ENGENHO- MSE:4598
inicia